In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/notebooks/swish9/social-networks-ads/__results__.html
/kaggle/input/notebooks/swish9/social-networks-ads/__resultx__.html
/kaggle/input/notebooks/swish9/social-networks-ads/__notebook__.ipynb
/kaggle/input/notebooks/swish9/social-networks-ads/__output__.json
/kaggle/input/notebooks/swish9/social-networks-ads/custom.css
/kaggle/input/datasets/muntahapolaris/dataset/Social_Network_Ads.csv


,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0
...,...,...,...
395,46,41000,1
396,51,23000,1
397,50,20000,1
398,36,33000,0


In [11]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
from matplotlib import pyplot as plt
%matplotlib inline

In [12]:
import pandas as pd
df = pd.read_csv("/kaggle/input/datasets/muntahapolaris/dataset/Social_Network_Ads.csv")
df

,Age,EstimatedSalary,Purchased
0,19,19000,0
1,35,20000,0
2,26,43000,0
3,27,57000,0
4,19,76000,0
...,...,...,...
395,46,41000,1
396,51,23000,1
397,50,20000,1
398,36,33000,0


In [14]:
# Here we do the split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df[['Age', 'EstimatedSalary']], df.Purchased, test_size = 0.2, random_state = 25)

In [15]:
max_values = X_train.max();

X_train_scaled = X_train.copy();
X_train_scaled = X_train_scaled/max_values;
X_test_scaled = X_test.copy();
X_test_scaled = X_test_scaled/max_values;

**We Are Building Neural Network in Plain Python in here**

In [16]:
# this take an arrray Ar
def sigmoid (Ar):
    return 1/(1+np.exp(-Ar));
sigmoid(np.array([12,3,4]))

array([0.99999386, 0.95257413, 0.98201379])

In [26]:
# Implementing the log_loss
def log_loss(y_true, y_predicted):
    epsilon = 1e-15  # a value extremely close to 0
    
    # This replaces both max and min loops safely in 1 line
    y_pred_new = np.clip(y_predicted, epsilon, 1 - epsilon)
    
    # Don't forget to calculate and return the actual log loss formula here!
    loss = -np.mean(y_true * np.log(y_pred_new) + (1 - y_true) * np.log(1 - y_pred_new))
    return loss
    

In [29]:
class NN:
    def __init__ (self):
        self.w1 = 1;
        self.w2 = 1;
        self.bias = 0;
        

    def gradient_descent(self,age, EstimatedSalary, y_true, epochs, loss_threshold ):
        w1 = w2 = 1; bias = 0; rate = 0.5; n= len(age)
        for i in range(epochs):
            weighted_sum = w1 * age + w2 * EstimatedSalary + bias;
            y_predicted = sigmoid(weighted_sum)
            loss = log_loss(y_true, y_predicted)

            w1_derivative =  (1/n)*np.dot(np.transpose(age), (y_predicted - y_true));
            w2_derivative = (1/n)*np.dot(np.transpose(EstimatedSalary), (y_predicted - y_true));
            bias_derivative = np.mean(y_predicted - y_true);

            w1 = w1 - rate*w1_derivative;
            w2 = w2 - rate*w2_derivative;
            bias =bias - rate*bias_derivative;

            if loss <= loss_threshold:
                break;
        return w1, w2, bias;

            
            
        
    def fit(self, X, y, epoch, loss_threshold):
        self.w1, self.w2, self.bias = self.gradient_descent(X['Age'], X['EstimatedSalary'], y, epoch, loss_threshold);
        print(f"Final weights and bias: w1: {self.w1}, w2: {self.w2}, bias: {self.bias}")
        
    def predict(self, X_test):
        weighted_sum = self.w1 * X_test['Age'] + self.w2 * X_test['EstimatedSalary'] + self.bias;
        return sigmoid(weighted_sum);
        
    

In [30]:
# Testing our custom model that we built from scratch
ourModel = NN()
ourModel.fit(X_train_scaled, y_train, 200, 0.5)


Final weights and bias: w1: 2.5711782181459424, w2: 1.6726111331959737, bias: -2.921776760350643


In [31]:
ourModel.predict(X_test_scaled)

121    0.369733
162    0.275231
338    0.336245
375    0.355783
262    0.696139
         ...   
117    0.310190
12     0.248663
102    0.356287
56     0.197670
260    0.362781
Length: 80, dtype: float64


**In Case we wanna use Keras and Tensorflow to create our neural Network: We May Simply**


In [32]:
model = keras.Sequential([
    keras.layers.Dense(1, input_shape=(2,), activation='sigmoid', kernel_initializer='ones', bias_initializer='zeros')
])
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(X_train_scaled, y_train, epochs=5000)

Epoch 1/5000


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-07-20 19:00:02.259804: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.3625 - loss: 0.9000  
Epoch 2/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8933 
Epoch 3/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8862 
Epoch 4/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8796 
Epoch 5/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8727 
Epoch 6/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8662 
Epoch 7/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8599 
Epoch 8/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8534 
Epoch 9/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8472 
Epoch 10/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8411 
Epoch 11/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3625 - loss: 0.8350 
Epoch 12/5000
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 

KeyboardInterrupt: 